# 🔧 Notebook 03 — Feature Engineering & Data Preparation

## Obiettivo
Trasformare il dataset sintetico generato nel Notebook 02 in un **unico dataset**
pronto per l'addestramento del modello ML di Depression Prediction.

### Features Estratte (8 predittori temporali)
| Feature | Descrizione | Razionale Clinico |
|---|---|---|
| `valence_mean` | Media della Valence sui 14 giorni | Tono emotivo complessivo (Kuppens et al., 2010) |
| `valence_std` | Deviazione standard della Valence | Instabilità emotiva (Houben et al., 2015) |
| `valence_ema_3d` | Media Mobile Esponenziale (3 giorni) | Trend recente con recency bias |
| `valence_trend_5d` | Pendenza lineare ultimi 5 giorni | Direzione del cambiamento (Wichers et al., 2016) |
| `max_neg_streak` | Max giorni consecutivi con valence < 0 | Persistenza depressiva (aan het Rot et al., 2012) |
| `missing_ratio` | Proporzione di giorni mancanti | Proxy di disengagement MNAR (Mohr et al., 2017) |
| `intensity_mean` | Media dell'intensità emotiva | Appiattimento affettivo (Rottenberg et al., 2005) |
| `dominant_mood_valence` | Valence dello stato d'animo più frequente | Pattern emotivo prevalente |

### Target
`phq9_total` — Punteggio continuo del questionario PHQ-9 (0-27), regressione.

## 1. Import e Caricamento

In [1]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', '..', '..', 'docs', 'latex', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Caricamento dataset sintetico dal Notebook 02
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'synthetic_depression_data_v2.csv'))
print(f"Dataset caricato: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(f"Pazienti: {df['patient_id'].nunique():,}")
print(f"Missing records: {df['is_missing'].sum():,} ({df['is_missing'].mean()*100:.1f}%)")

# Caricamento mappa V-A
with open(os.path.join(PROCESSED_DIR, 'statistical_patterns.json'), 'r', encoding='utf-8') as f:
    patterns = json.load(f)
VA_MAP = patterns['sintonia_valence_arousal_map']

print(f"\n✅ Dati pronti per il Feature Engineering")

Dataset caricato: 1,400,000 righe × 10 colonne
Pazienti: 100,000
Missing records: 147,777 (10.6%)

✅ Dati pronti per il Feature Engineering


---
## 2. Feature Engineering

### Filosofia
Le features devono catturare **pattern temporali** e **dinamiche emotive**
che la letteratura scientifica associa alla depressione:
- **Tono medio basso** (Kuppens et al., 2010)
- **Alta instabilità emotiva** (Houben et al., 2015)
- **Trend discendente** (Wichers et al., 2016)
- **Streak negative prolungate** (aan het Rot et al., 2012)
- **Mancata risposta ai check-in** (Mohr et al., 2017)

In [2]:
def extract_features(patient_data):
    """Estrae un vettore di 8 features da 14 giorni di check-in emotivi."""
    pid = patient_data['patient_id'].iloc[0]
    phq9 = patient_data['phq9_total'].iloc[0]
    severity = patient_data['severity'].iloc[0]

    present = patient_data[~patient_data['is_missing']]
    n_total = len(patient_data)
    n_present = len(present)
    n_missing = n_total - n_present

    if n_present < 3:
        return None

    valences = present['valence'].values
    intensities = present['intensity'].values

    # Feature 1: Valence Media
    valence_mean = np.mean(valences)

    # Feature 2: Volatilità Emotiva (Houben et al., 2015)
    valence_std = np.std(valences) if n_present > 1 else 0.0

    # Feature 3: EMA ultimi 3 punti (recency bias)
    alpha = 2 / (3 + 1)
    ema = valences[0]
    for v in valences[1:]:
        ema = alpha * v + (1 - alpha) * ema
    valence_ema_3d = ema

    # Feature 4: Trend ultimi 5 punti (Wichers et al., 2016)
    if n_present >= 5:
        last5 = valences[-5:]
        x = np.arange(len(last5))
        slope, _ = np.polyfit(x, last5, 1)
        valence_trend_5d = slope
    elif n_present >= 2:
        x = np.arange(n_present)
        slope, _ = np.polyfit(x, valences, 1)
        valence_trend_5d = slope
    else:
        valence_trend_5d = 0.0

    # Feature 5: Max Streak Negativa (aan het Rot et al., 2012)
    max_neg = 0
    current_neg = 0
    for is_missing, valence in zip(patient_data['is_missing'].values,
                                    patient_data['valence'].values):
        if not is_missing and valence < 0:
            current_neg += 1
            max_neg = max(max_neg, current_neg)
        elif not is_missing:
            current_neg = 0

    # Feature 6: Missing Ratio (Mohr et al., 2017)
    missing_ratio = n_missing / n_total

    # Feature 7: Intensità Media (Rottenberg et al., 2005)
    intensity_mean = np.mean(intensities)

    # Feature 8: Stato Dominante (Russell, 1980)
    mood_counts = present['mood_state'].value_counts()
    dominant_mood = mood_counts.index[0] if len(mood_counts) > 0 else 'Neutro'
    dominant_mood_valence = VA_MAP.get(dominant_mood, {'valence': 0})['valence']

    return {
        'patient_id': pid,
        'valence_mean': round(valence_mean, 6),
        'valence_std': round(valence_std, 6),
        'valence_ema_3d': round(valence_ema_3d, 6),
        'valence_trend_5d': round(valence_trend_5d, 6),
        'max_neg_streak': int(max_neg),
        'missing_ratio': round(missing_ratio, 6),
        'intensity_mean': round(intensity_mean, 6),
        'dominant_mood_valence': round(dominant_mood_valence, 6),
        'phq9_total': int(phq9),
        'severity': severity
    }

In [3]:
print("🔧 Estrazione features in corso...")

features_list = []
skipped = 0

for pid, group in df.groupby('patient_id'):
    feat = extract_features(group)
    if feat is not None:
        features_list.append(feat)
    else:
        skipped += 1

features_df = pd.DataFrame(features_list)
print(f"\n✅ Feature Engineering completato!")
print(f"   Pazienti processati: {len(features_df):,}")
print(f"   Pazienti scartati (dati insufficienti): {skipped:,}")
print(f"\nShape della matrice features: {features_df.shape}")

🔧 Estrazione features in corso...



✅ Feature Engineering completato!
   Pazienti processati: 100,000
   Pazienti scartati (dati insufficienti): 0

Shape della matrice features: (100000, 11)


In [4]:
feature_cols = ['valence_mean', 'valence_std', 'valence_ema_3d', 'valence_trend_5d',
                'max_neg_streak', 'missing_ratio', 'intensity_mean', 'dominant_mood_valence']

print("\n═" * 70)
print("STATISTICHE DESCRITTIVE DELLE FEATURES ESTRATTE")
print("═" * 70)
print(features_df[feature_cols].describe().round(4).to_string())


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
STATISTICHE DESCRITTIVE DELLE FEATURES ESTRATTE
══════════════════════════════════════════════════════════════════════
       valence_mean  valence_std  valence_ema_3d  valence_trend_5d  max_neg_streak  missing_ratio  intensity_mean  dominant_mood_valence
count   100000.0000  100000.0000     100000.0000       100000.0000     100000.0000    100000.0000     100000.0000            100000.0000
mean         0.1564       0.3107          0.1433           -0.0001          2.5505         0.1056          4.9645                 0.0586
std          0.2384       0.0866          0.3104            0.1230          2.0963         0.1198          0.7972                 0.2910
min         -1.0000       0.0000         -1.0000           -0.5579          0.0000         0.0000          1.0000                -0.8000
25%          0.0536       0.2497         -0.0289       

---
## 3. Analisi delle Correlazioni

In [5]:
corr_cols = feature_cols + ['phq9_total']
corr_matrix = features_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlazione di Pearson'})
ax.set_title('Matrice di Correlazione: Features vs PHQ-9', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dp_feature_correlation.png'), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figura salvata: dp_feature_correlation.png")

# Verifica data leakage
print("\n═" * 60)
print("CORRELAZIONE FEATURES ↔ PHQ-9 (Target)")
print("═" * 60)
phq_corrs = corr_matrix['phq9_total'].drop('phq9_total').sort_values()
for feat, corr_val in phq_corrs.items():
    bar = '█' * int(abs(corr_val) * 30)
    flag = ' ⚠️' if abs(corr_val) > 0.90 else ''
    print(f"  {feat:<25} {corr_val:>+.4f}  {bar}{flag}")

✅ Figura salvata: dp_feature_correlation.png

═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
CORRELAZIONE FEATURES ↔ PHQ-9 (Target)
════════════════════════════════════════════════════════════
  valence_mean              -0.6165  ██████████████████
  valence_ema_3d            -0.4171  ████████████
  dominant_mood_valence     -0.3151  █████████
  intensity_mean            -0.2075  ██████
  valence_std               -0.0288  
  valence_trend_5d          +0.0047  
  max_neg_streak            +0.4308  ████████████
  missing_ratio             +0.6966  ████████████████████


---
## 4. Standardizzazione e Preparazione del Dataset Finale

### Target Regressivo
L'obiettivo del modello è predire l'esatto punteggio PHQ-9 (0-27),
non una classificazione binaria. Questo garantisce granularità clinica
superiore nel monitoraggio (Kroenke et al., 2001).

In [6]:
y = features_df['phq9_total'].values

print(f"\n═" * 60)
print(f"DISTRIBUZIONE TARGET REGRESSIONE (PHQ-9)")
print(f"═" * 60)
print(f"  Range Punteggi:  {y.min()} - {y.max()}")
print(f"  Media:           {y.mean():.2f}")
print(f"  Mediana:         {np.median(y):.0f}")
print(f"  Dev. Standard:   {y.std():.2f}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
DISTRIBUZIONE TARGET REGRESSIONE (PHQ-9)
════════════════════════════════════════════════════════════
  Range Punteggi:  0 - 27
  Media:           5.83
  Mediana:         4
  Dev. Standard:   5.47


In [7]:
# Standardizzazione delle 8 features
X = features_df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n═" * 60)
print(f"STANDARDIZZAZIONE COMPLETATA")
print(f"═" * 60)
print(f"  Shape X_scaled: {X_scaled.shape}")
print(f"  Shape y:        {y.shape}")
print(f"\n  Verifica post-scaling (media ≈ 0, std ≈ 1):")
for i, col in enumerate(feature_cols):
    print(f"    {col:<25}  μ = {X_scaled[:, i].mean():>+.6f}  σ = {X_scaled[:, i].std():.6f}")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
STANDARDIZZAZIONE COMPLETATA
════════════════════════════════════════════════════════════
  Shape X_scaled: (100000, 8)
  Shape y:        (100000,)

  Verifica post-scaling (media ≈ 0, std ≈ 1):
    valence_mean               μ = -0.000000  σ = 1.000000
    valence_std                μ = +0.000000  σ = 1.000000
    valence_ema_3d             μ = +0.000000  σ = 1.000000
    valence_trend_5d           μ = +0.000000  σ = 1.000000
    max_neg_streak             μ = +0.000000  σ = 1.000000
    missing_ratio              μ = -0.000000  σ = 1.000000
    intensity_mean             μ = +0.000000  σ = 1.000000
    dominant_mood_valence      μ = +0.000000  σ = 1.000000


---
## 5. Salvataggio Dataset Finale Unificato

Un unico file CSV contenente le 8 feature scalate + il target `phq9_total`.
Questo è il file che verrà caricato direttamente dal modello di ML.

In [8]:
# Dataset unificato: features scalate + target
final_df = pd.DataFrame(X_scaled, columns=feature_cols)
final_df['phq9_total'] = y

output_path = os.path.join(PROCESSED_DIR, 'depression_model_dataset.csv')
final_df.to_csv(output_path, index=False)

print(f"\n✅ Dataset finale unificato salvato:")
print(f"   📄 {output_path}")
print(f"   Shape: {final_df.shape[0]:,} × {final_df.shape[1]} (8 features + 1 target)")
print(f"\nPrime 5 righe:")
print(final_df.head().to_string())

# Salva anche il dataset completo con metadati (per analisi)
features_df.to_csv(os.path.join(PROCESSED_DIR, 'features_complete.csv'), index=False)

print(f"\n   📄 features_complete.csv (dataset completo con metadati, per analisi)")


✅ Dataset finale unificato salvato:
   📄 ..\data\processed\depression_model_dataset.csv
   Shape: 100,000 × 9 (8 features + 1 target)

Prime 5 righe:
   valence_mean  valence_std  valence_ema_3d  valence_trend_5d  max_neg_streak  missing_ratio  intensity_mean  dominant_mood_valence  phq9_total
0     -0.393691     1.020105       -0.791685         -1.538304        0.214406       0.311464        0.567221              -0.201301           1
1      1.008918     0.735014        1.118160          1.692617       -0.262624      -0.881366        0.940548               2.719966           2
2     -0.041412    -0.622969       -0.643712         -0.708032        0.214406      -0.284947        0.044562              -0.201301           0
3     -0.228420     0.568414       -1.204581          0.194030        1.645497      -0.284947        0.430525              -0.201301           7
4      1.245983    -0.482422        0.421489         -0.017847       -1.216685      -0.284947        0.719998              -


   📄 features_complete.csv (dataset completo con metadati, per analisi)


In [9]:
from sklearn.model_selection import train_test_split

print("✂️ Suddivisione Dataset in Train (80%) e Test (20%)...")

# Carichiamo il dataset appena salvato (o usiamo final_df)
# Per sicurezza usiamo final_df che è già in memoria
train_df, test_df = train_test_split(
    final_df, test_size=0.20, random_state=42, shuffle=True
)

train_path = os.path.join(PROCESSED_DIR, 'train.csv')
test_path = os.path.join(PROCESSED_DIR, 'test.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"   ✅ Train set salvato in: {train_path} ({len(train_df):,} record)")
print(f"   ✅ Test set salvato in:  {test_path} ({len(test_df):,} record)")

✂️ Suddivisione Dataset in Train (80%) e Test (20%)...


   ✅ Train set salvato in: ..\data\processed\train.csv (80,000 record)
   ✅ Test set salvato in:  ..\data\processed\test.csv (20,000 record)


In [10]:
# ═══════════════════════════════════════════════════════════════
# VISUALIZZAZIONE DISTRIBUZIONE TARGET (PER TESI)
# ═══════════════════════════════════════════════════════════════
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
# Plotting KDE for Training and Test PHQ-9 distribution
sns.kdeplot(train_df['phq9_total'], label='Training Set (80%)', fill=True, color='royalblue', alpha=0.3)
sns.kdeplot(test_df['phq9_total'], label='Test Set (20%)', fill=True, color='crimson', alpha=0.3)

plt.title('Distribuzione PHQ-9: Training vs Test Set (Dataset Sintetico)', fontsize=14, fontweight='bold')
plt.xlabel('Punteggio PHQ-9 Total', fontsize=12)
plt.ylabel('Densità', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Save the figure to the shared latex directory
fig_path = os.path.join('../../../docs/latex/figures', 'dp_target_distribution.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\n📸 Grafico distribuzione target salvato per LaTeX in: {fig_path}")
plt.show()


📸 Grafico distribuzione target salvato per LaTeX in: ../../../docs/latex/figures\dp_target_distribution.png


---
## 6. Riepilogo Finale

### Pipeline completata:
```
Dataset Reali (Student Life + 14-Day)
   ↓ Notebook 01: Analisi e Mappatura V-A
Pattern Statistici (JSON) — ruolo: varianza e transizioni
   ↓ Notebook 02: Generazione Sintetica (Literature-Driven)
100K Pazienti × 14 Giorni (CSV)
   ↓ Notebook 03: Feature Engineering + Scaling
Dataset Finale Unificato (8 features + target PHQ-9)
   → Pronto per il modello ML di regressione
```